# Helmet Detection — Retraining on Roboflow Dataset (Colab)

Dataset: `bac_hien_construction_safety_2024` (v8) — classes: `Helmet`, `No-Helmet`, `Person`

This notebook fixes the common reasons a first training run underperforms:
- **Small objects** (helmets on heads) need a *larger* image size than the default 640, or the model downsamples them into a handful of pixels before it ever sees them.
- **Class imbalance** — if your dataset has far more `Person` boxes than `No-Helmet` boxes, the model will be biased toward predicting the majority class. This notebook checks that up front.
- **Too few epochs / no augmentation** — small custom datasets need stronger augmentation and enough epochs (with early stopping) to actually converge.
- **Wrong model size** — nano models are fast but often too weak for anything beyond very easy detections. We default to a mid-sized model here.

## 1. Setup

In [ ]:
!nvidia-smi

from google.colab import drive
drive.mount('/content/drive')

!pip install -q roboflow ultralytics

## 2. Download the dataset from Roboflow

In [ ]:
import os
os.chdir('/content')

from roboflow import Roboflow

rf = Roboflow(api_key="R9PSIw7tIYNb2vd8QwWq")
project = rf.workspace("constructionsatefy").project("bac_hien_construction_safety_2024")
version = project.version(8)
dataset = version.download("yolov11")  # yolov11/yolov8/yolov12 all produce the same label format

print('Dataset downloaded to:', dataset.location)
print(os.listdir(dataset.location))

In [ ]:
import glob

yaml_candidates = glob.glob(os.path.join(dataset.location, '**', 'data.yaml'), recursive=True)
assert yaml_candidates, 'Could not find data.yaml in the downloaded dataset.'
yaml_path = yaml_candidates[0]

with open(yaml_path) as f:
    print(f.read())

## 3. Check class balance BEFORE training
If one class massively outnumbers another, that imbalance is very likely part of why the model performs poorly on the minority class (often `No-Helmet`, since violations are usually rarer in real photos than compliant workers).

In [ ]:
import yaml
from collections import Counter

with open(yaml_path) as f:
    data_cfg = yaml.safe_load(f)

class_names = data_cfg['names']
print('Classes:', class_names)

counts = Counter()
for split in ['train', 'val', 'test']:
    split_path = data_cfg.get(split)
    if not split_path:
        continue
    labels_dir = split_path.replace('images', 'labels')
    # Roboflow's exported path may be relative to dataset.location
    if not os.path.isabs(labels_dir):
        labels_dir = os.path.join(dataset.location, labels_dir)
    if not os.path.isdir(labels_dir):
        continue
    for label_file in glob.glob(os.path.join(labels_dir, '*.txt')):
        with open(label_file) as lf:
            for line in lf:
                if line.strip():
                    cls_id = int(line.split()[0])
                    counts[class_names[cls_id]] += 1

print('\nInstance counts per class (across train/val/test):')
for name, count in counts.items():
    print(f'  {name}: {count}')

total = sum(counts.values())
print('\nClass balance:')
for name, count in counts.items():
    print(f'  {name}: {100 * count / total:.1f}%')

## 4. Train a pretrained YOLO11 model

Key changes from a naive first attempt:
- `imgsz=960` instead of 640 — helmets are small relative to the full image, so a higher input resolution helps the model actually resolve them instead of downsampling them away.
- `yolo11m.pt` (medium) instead of nano/x — a solid accuracy/speed balance for a 3-class problem on a free Colab GPU. Swap to `yolo11l.pt` if you have time/GPU budget for more accuracy, or `yolo11s.pt` if training is too slow.
- More epochs (150) with `patience=30` (early stopping) — small custom datasets typically need more epochs than COCO-scale fine-tuning.
- Explicit augmentation tuned for small-object detection: increased `scale`/`translate` jitter, `mosaic` kept on, `mixup` added lightly.
- `cos_lr=True` for a smoother learning-rate decay, which tends to help small datasets converge better than the default step schedule.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11m.pt')  # pretrained on COCO

results = model.train(
    data=yaml_path,
    epochs=150,
    imgsz=960,
    batch=16,
    patience=30,
    cos_lr=True,
    optimizer='AdamW',
    lr0=0.001,
    scale=0.6,       # stronger scale jitter — helps generalize to varying helmet sizes
    translate=0.15,
    mixup=0.1,
    mosaic=1.0,
    hsv_h=0.02, hsv_s=0.7, hsv_v=0.4,  # color jitter for varying lighting on construction sites
    degrees=5.0,     # slight rotation jitter
    project='/content/runs',
    name='helmet_yolo11m_v2'
)

## 5. Validate and inspect per-class performance

In [ ]:
best_weights = '/content/runs/helmet_yolo11m_v2/weights/best.pt'
trained_model = YOLO(best_weights)
metrics = trained_model.val(data=yaml_path)

print('mAP50-95:', metrics.box.map)
print('mAP50:', metrics.box.map50)
print('\nPer-class mAP50:')
for i, name in enumerate(class_names):
    print(f'  {name}: {metrics.box.maps[i]:.3f}')

In [ ]:
# Confusion matrix and PR curves are saved automatically by val() — display them
from IPython.display import Image, display
import glob as _glob

val_dir = trained_model.trainer.save_dir if hasattr(trained_model, 'trainer') else metrics.save_dir
for img_path in _glob.glob(os.path.join(str(val_dir), '*.png')):
    print(os.path.basename(img_path))
    display(Image(filename=img_path, width=600))

## 6. Run inference on a few validation images

In [ ]:
import random

val_images_dir = data_cfg['val'] if os.path.isabs(data_cfg['val']) else os.path.join(dataset.location, data_cfg['val'])
sample_paths = random.sample(os.listdir(val_images_dir), min(4, len(os.listdir(val_images_dir))))

for fname in sample_paths:
    img_path = os.path.join(val_images_dir, fname)
    res = trained_model.predict(img_path, save=True, conf=0.25)
    display(Image(filename=res[0].save_dir + '/' + fname, width=700))

## 7. Save trained weights back to Drive

In [ ]:
import shutil

SAVE_DIR = '/content/drive/MyDrive/helmet_detection_v2'
os.makedirs(SAVE_DIR, exist_ok=True)
shutil.copy(best_weights, os.path.join(SAVE_DIR, 'best.pt'))
shutil.copy(yaml_path, os.path.join(SAVE_DIR, 'data.yaml'))
print('Saved to', SAVE_DIR)